# 04 - Démo du Pipeline TTS : Synthèse Vocale en Arabe Hassaniya

**Projet :** Système de Synthèse Vocale pour l'Arabe Hassaniya par Apprentissage par Transfert
**Auteur :** Mohamed Salem Ebnou Echvagha Oubeid (C34613)
**Module :** Dialectes NLP — Master M1 IA
**Date :** Juin 2026

---

## Objectif

Ce notebook démontre un **pipeline TTS complet** pour l'arabe hassaniya en utilisant
l'**apprentissage par transfert** à partir de modèles pré-entraînés. Notre approche :

1. **Référence** : Utiliser gTTS (Google Text-to-Speech) pour la synthèse en arabe
2. **Architecture** : Expliquer l'architecture Tacotron2/VITS que nous affinerions
3. **Comparaison** : Comparer les enregistrements originaux vs. la parole synthétisée
4. **Analyse** : Évaluer la qualité par l'analyse spectrale

### Pourquoi l'Apprentissage par Transfert ?

Entraîner un modèle TTS à partir de zéro nécessite :
- **10 000+ heures** de parole enregistrée
- **Plusieurs jours GPU** d'entraînement
- **Des enregistrements studio professionnels** de qualité constante

Avec seulement **294 échantillons**, nous utilisons l'apprentissage par transfert :
- Partir d'un modèle pré-entraîné sur l'arabe (ou multilingue)
- Affiner sur notre petit jeu de données hassaniya
- Adapter les connaissances arabes du modèle à notre dialecte spécifique

## 1. Configuration

In [ ]:
# Installer les dépendances TTS
# !pip install gTTS librosa soundfile matplotlib pandas pyarrow numpy torch torchaudio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import os

import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display, HTML

# Moteur TTS
try:
    from gtts import gTTS
    GTTS_AVAILABLE = True
    print('gTTS chargé avec succès')
except ImportError:
    GTTS_AVAILABLE = False
    print('gTTS non disponible — installer avec : pip install gTTS')

# PyTorch
try:
    import torch
    import torchaudio
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} chargé')
    print(f'CUDA disponible : {torch.cuda.is_available()}')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch non disponible')

SAMPLE_RATE = 22050
plt.rcParams['figure.figsize'] = (14, 4)
sns.set_style('whitegrid')

os.makedirs('../results/generated_audio', exist_ok=True)
os.makedirs('../results/spectrograms', exist_ok=True)
print('\nConfiguration terminée')

In [ ]:
# Charger le jeu de données
df = pd.read_parquet('../audios_dataset.parquet')
print(f'Jeu de données : {len(df)} échantillons')

## 2. Vue d'Ensemble de l'Architecture TTS

### Pipeline TTS Moderne

Un système TTS complet comporte deux composants principaux :

```
Texte → [Encodeur de Texte] → [Décodeur] → Mel Spectrogramme → [Vocodeur] → Forme d'Onde Audio
```

#### Composant 1 : Modèle Acoustique (Texte → Mel Spectrogramme)

| Modèle | Année | Description |
|--------|-------|-------------|
| **Tacotron** | 2017 | Seq2seq avec attention, entrée par caractères |
| **Tacotron2** | 2018 | Attention améliorée, meilleure qualité |
| **FastSpeech2** | 2020 | Non-autorégressif, inférence plus rapide |
| **VITS** | 2021 | Bout-en-bout, pas de vocodeur séparé nécessaire |

#### Composant 2 : Vocodeur (Mel Spectrogramme → Forme d'Onde)

| Modèle | Année | Description |
|--------|-------|-------------|
| **WaveNet** | 2016 | Autorégressif, très lent |
| **WaveGlow** | 2018 | Basé sur les flux, génération parallèle |
| **HiFi-GAN** | 2020 | Basé sur les GANs, rapide et haute qualité |

### Notre Approche d'Apprentissage par Transfert

```
Modèle Arabe Pré-entraîné (grand jeu de données)
        ↓
Geler les couches de l'encodeur (conserver les connaissances arabes)
        ↓  
Affiner le décodeur sur les données hassaniya (294 échantillons)
        ↓
Modèle TTS Adapté au Hassaniya
```

## 3. Référence : TTS Arabe avec gTTS

Nous commençons avec **Google Text-to-Speech (gTTS)** comme système de référence.
gTTS prend en charge l'arabe (ASM) — bien qu'il ne produise pas de la parole avec l'accent hassaniya,
il démontre le pipeline TTS et nous donne un point de comparaison.

In [ ]:
def synthesize_gtts(text, output_path, lang='ar'):
    """Synthétiser la parole avec Google TTS."""
    if not GTTS_AVAILABLE:
        print('gTTS non disponible')
        return None
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        tts.save(output_path)
        audio, sr = librosa.load(output_path, sr=SAMPLE_RATE)
        return audio
    except Exception as e:
        print(f'Erreur : {e}')
        return None

# Phrases de test — expressions en arabe hassaniya
test_sentences = [
    df['transcription'].iloc[0],   # Premier échantillon du jeu de données
    df['transcription'].iloc[5],   # Un autre échantillon
    df['transcription'].iloc[10],  # Un autre échantillon
    'السلام عليكم ورحمة الله',       # Salutation courante
    'كيف حالك اليوم',               # Comment vas-tu aujourd'hui
]

print('Phrases de test pour la synthèse :')
for i, sent in enumerate(test_sentences):
    print(f'  [{i}] {sent}')

In [ ]:
# Générer la parole pour les phrases de test
generated_audios = []

if GTTS_AVAILABLE:
    for i, text in enumerate(test_sentences):
        output_path = f'../results/generated_audio/generated_{i:02d}.mp3'
        print(f'\nGénération [{i}] : "{text}"')
        audio = synthesize_gtts(text, output_path)
        if audio is not None:
            generated_audios.append(audio)
            print(f'  Durée : {len(audio)/SAMPLE_RATE:.2f}s')
            display(Audio(audio, rate=SAMPLE_RATE))
        else:
            generated_audios.append(None)
    print(f'\n{sum(1 for a in generated_audios if a is not None)} échantillons audio générés')
else:
    print('gTTS non disponible — synthèse ignorée')
    print('Installer avec : pip install gTTS')

## 4. Comparaison : Original vs. Synthétisé

Comparons les enregistrements originaux hassaniya avec la parole générée par gTTS.
Cela met en évidence la différence entre la **parole dialectale naturelle** et
la **TTS en arabe standard** — motivant le besoin de modèles spécifiques au dialecte.

In [ ]:
def decode_original(audio_dict, sr=SAMPLE_RATE):
    """Décoder l'audio original du jeu de données."""
    audio_bytes = audio_dict.get('bytes', b'')
    try:
        audio, orig_sr = sf.read(io.BytesIO(audio_bytes))
        audio = audio.astype(np.float32)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        if orig_sr != sr:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
        return audio
    except:
        return None

# Comparer le premier échantillon : original vs généré
original_audio = decode_original(df['audio'].iloc[0])

if original_audio is not None:
    print(f'Texte : "{df["transcription"].iloc[0]}"')
    print(f'\nEnregistrement original hassaniya :')
    display(Audio(original_audio, rate=SAMPLE_RATE))
    
    if generated_audios and generated_audios[0] is not None:
        print(f'\ngTTS généré (Arabe Standard) :')
        display(Audio(generated_audios[0], rate=SAMPLE_RATE))

In [ ]:
# Comparaison visuelle : spectrogrammes
if original_audio is not None and generated_audios and generated_audios[0] is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    
    # Forme d'onde originale
    t_orig = np.linspace(0, len(original_audio)/SAMPLE_RATE, len(original_audio))
    axes[0, 0].plot(t_orig, original_audio, color='#1565C0', linewidth=0.5)
    axes[0, 0].set_title('Original Hassaniya — Forme d\'Onde', fontweight='bold')
    axes[0, 0].set_ylabel('Amplitude')
    
    # Spectrogramme original
    mel_orig = librosa.feature.melspectrogram(y=original_audio, sr=SAMPLE_RATE, n_mels=80)
    mel_orig_db = librosa.power_to_db(mel_orig, ref=np.max)
    librosa.display.specshow(mel_orig_db, y_axis='mel', x_axis='time',
                             sr=SAMPLE_RATE, hop_length=256, ax=axes[0, 1])
    axes[0, 1].set_title('Original Hassaniya — Mel Spectrogramme', fontweight='bold')
    
    # Forme d'onde générée
    gen_audio = generated_audios[0]
    t_gen = np.linspace(0, len(gen_audio)/SAMPLE_RATE, len(gen_audio))
    axes[1, 0].plot(t_gen, gen_audio, color='#E65100', linewidth=0.5)
    axes[1, 0].set_title('gTTS Généré (Arabe Standard) — Forme d\'Onde', fontweight='bold')
    axes[1, 0].set_xlabel('Temps (s)')
    axes[1, 0].set_ylabel('Amplitude')
    
    # Spectrogramme généré
    mel_gen = librosa.feature.melspectrogram(y=gen_audio, sr=SAMPLE_RATE, n_mels=80)
    mel_gen_db = librosa.power_to_db(mel_gen, ref=np.max)
    librosa.display.specshow(mel_gen_db, y_axis='mel', x_axis='time',
                             sr=SAMPLE_RATE, hop_length=256, ax=axes[1, 1])
    axes[1, 1].set_title('gTTS Généré — Mel Spectrogramme', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../results/spectrograms/original_vs_generated.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Comparaison sauvegardée dans results/spectrograms/original_vs_generated.png')
else:
    print('Impossible de créer la comparaison — données audio manquantes')

## 5. Architecture d'Apprentissage par Transfert

Ci-dessous, nous définissons l'architecture qui serait utilisée pour affiner un
modèle TTS pré-entraîné sur les données hassaniya. Cela démontre l'**architecture du modèle**
même si l'entraînement complet nécessite plus de ressources de calcul que celles disponibles.

In [ ]:
if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn
    
    class HassaniyaTTSEncoder(nn.Module):
        """Encodeur de texte pour le TTS Hassaniya.
        
        Convertit les IDs de caractères en représentations cachées.
        Architecture inspirée de l'encodeur de Tacotron2.
        """
        def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, n_layers=2):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim)
            self.convolutions = nn.Sequential(
                nn.Conv1d(embed_dim, hidden_dim, kernel_size=5, padding=2),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.5),
            )
            self.lstm = nn.LSTM(hidden_dim, hidden_dim // 2, n_layers,
                                batch_first=True, bidirectional=True)
        
        def forward(self, x):
            x = self.embedding(x)           # [B, T] -> [B, T, E]
            x = x.transpose(1, 2)           # [B, E, T]
            x = self.convolutions(x)        # [B, H, T]
            x = x.transpose(1, 2)           # [B, T, H]
            x, _ = self.lstm(x)             # [B, T, H]
            return x
    
    
    class HassaniyaTTSDecoder(nn.Module):
        """Décodeur simple de Mel spectrogramme.
        
        Prend les sorties de l'encodeur et génère des trames de Mel spectrogramme.
        Version simplifiée pour la démonstration.
        """
        def __init__(self, hidden_dim=256, n_mels=80):
            super().__init__()
            self.lstm = nn.LSTM(hidden_dim + n_mels, hidden_dim, 2, batch_first=True)
            self.mel_projection = nn.Linear(hidden_dim, n_mels)
            self.stop_projection = nn.Linear(hidden_dim, 1)
        
        def forward(self, encoder_output):
            B, T, H = encoder_output.shape
            mel_input = torch.zeros(B, T, 80).to(encoder_output.device)
            combined = torch.cat([encoder_output, mel_input], dim=-1)
            output, _ = self.lstm(combined)
            mel_output = self.mel_projection(output)
            stop_token = torch.sigmoid(self.stop_projection(output))
            return mel_output, stop_token
    
    
    class HassaniyaTTS(nn.Module):
        """Modèle TTS Hassaniya complet combinant encodeur et décodeur."""
        def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, n_mels=80):
            super().__init__()
            self.encoder = HassaniyaTTSEncoder(vocab_size, embed_dim, hidden_dim)
            self.decoder = HassaniyaTTSDecoder(hidden_dim, n_mels)
        
        def forward(self, text_ids):
            encoder_output = self.encoder(text_ids)
            mel_output, stop_token = self.decoder(encoder_output)
            return mel_output, stop_token
    
    # Instancier le modèle
    VOCAB_SIZE = 50  # vocabulaire approximatif de caractères hassaniya
    model = HassaniyaTTS(vocab_size=VOCAB_SIZE)
    
    # Compter les paramètres
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print('=== Architecture du Modèle TTS Hassaniya ===')
    print(f'Paramètres totaux : {total_params:,}')
    print(f'Paramètres entraînables : {trainable_params:,}')
    print(f'Taille du modèle : ~{total_params * 4 / 1024 / 1024:.1f} Mo (float32)')
    print()
    print(model)
else:
    print('PyTorch non disponible — description de l\'architecture uniquement')
    print()
    print('Architecture TTS Hassaniya :')
    print('  Encodeur : Embedding → 3x Conv1D → BiLSTM')
    print('  Décodeur : LSTM → Linéaire → Mel Spectrogramme')
    print('  Vocodeur : HiFi-GAN (pré-entraîné)')

In [ ]:
# Démonstration de la passe avant
if TORCH_AVAILABLE:
    dummy_input = torch.randint(0, VOCAB_SIZE, (1, 20))  # batch=1, seq_len=20
    
    with torch.no_grad():
        mel_output, stop_token = model(dummy_input)
    
    print('Démonstration de la passe avant :')
    print(f'  Forme d\'entrée :      {dummy_input.shape}  (batch, longueur_texte)')
    print(f'  Forme sortie Mel :    {mel_output.shape}  (batch, trames_temporelles, n_mels)')
    print(f'  Forme stop token :    {stop_token.shape}  (batch, trames_temporelles, 1)')
    
    # Visualiser la sortie Mel factice
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(mel_output[0].T.numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_xlabel('Trames Temporelles')
    ax.set_ylabel('Bandes Mel')
    ax.set_title('Sortie du Modèle (Non Entraîné) — Mel Spectrogramme Aléatoire')
    plt.colorbar(im, ax=ax, label='Amplitude')
    plt.tight_layout()
    plt.show()
    print('Note : Ceci est une sortie aléatoire d\'un modèle non entraîné.')
    print('Avec l\'affinage sur des données hassaniya, cela produirait des spectrogrammes significatifs.')

## 6. Stratégie d'Apprentissage par Transfert

### Processus d'Affinage

Pour un TTS hassaniya de qualité production, le processus d'affinage serait :

| Étape | Action | Couches |
|-------|--------|---------|
| 1 | Charger le modèle TTS arabe pré-entraîné | Toutes |
| 2 | Geler l'encodeur (préserver la phonologie arabe) | Encodeur |
| 3 | Affiner le décodeur sur les cibles Mel hassaniya | Décodeur |
| 4 | Dégeler progressivement les couches de l'encodeur | Encodeur (couches supérieures) |
| 5 | Affiner le vocodeur sur l'audio hassaniya | Vocodeur |

### Configuration d'Entraînement (Référence)

```python
# Hyperparamètres typiques d'affinage
config = {
    'learning_rate': 1e-4,        # LR bas pour l'affinage
    'batch_size': 16,
    'epochs': 100,
    'warmup_steps': 500,
    'grad_clip': 1.0,
    'weight_decay': 1e-6,
    'scheduler': 'cosine_annealing',
    'frozen_layers': ['encoder.embedding', 'encoder.convolutions'],
}
```

In [ ]:
# Démonstration de l'apprentissage par transfert : stratégie gel/dégel
if TORCH_AVAILABLE:
    # Simuler le gel de l'encodeur
    for param in model.encoder.parameters():
        param.requires_grad = False
    
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print('=== Configuration de l\'Apprentissage par Transfert ===')
    print(f'Paramètres gelés (encodeur) :      {frozen:,}')
    print(f'Paramètres entraînables (décodeur) : {trainable:,}')
    print(f'Ratio entraînable : {100*trainable/(frozen+trainable):.1f}%')
    print()
    print('Stratégie : N\'entraîner que le décodeur sur les données hassaniya')
    print('L\'encodeur conserve ses connaissances de la langue arabe')
    
    # Dégeler pour un état propre
    for param in model.encoder.parameters():
        param.requires_grad = True

## 7. Génération par Lots & Résultats

In [ ]:
# Générer la parole pour plus d'échantillons hassaniya
if GTTS_AVAILABLE:
    n_samples = min(10, len(df))
    
    print(f'Génération de la parole pour {n_samples} échantillons hassaniya...\n')
    
    for i in range(n_samples):
        text = df['transcription'].iloc[i]
        output_path = f'../results/generated_audio/hassaniya_gen_{i:03d}.mp3'
        
        try:
            tts = gTTS(text=text, lang='ar', slow=False)
            tts.save(output_path)
            audio, _ = librosa.load(output_path, sr=SAMPLE_RATE)
            duration = len(audio) / SAMPLE_RATE
            print(f'  [{i:2d}] "{text[:50]}..." → {duration:.1f}s')
        except Exception as e:
            print(f'  [{i:2d}] Erreur : {e}')
    
    print(f'\nTous les audios générés sauvegardés dans results/generated_audio/')

In [ ]:
# Grille de comparaison : plusieurs échantillons
if GTTS_AVAILABLE:
    n_compare = 4
    fig, axes = plt.subplots(n_compare, 2, figsize=(16, 3 * n_compare))
    
    for i in range(n_compare):
        # Original
        orig = decode_original(df['audio'].iloc[i])
        if orig is not None:
            mel_orig = librosa.feature.melspectrogram(y=orig, sr=SAMPLE_RATE, n_mels=80)
            mel_orig_db = librosa.power_to_db(mel_orig, ref=np.max)
            librosa.display.specshow(mel_orig_db, y_axis='mel', x_axis='time',
                                     sr=SAMPLE_RATE, hop_length=256, ax=axes[i, 0])
        text = df['transcription'].iloc[i]
        axes[i, 0].set_title(f'Original : "{text[:35]}..."', fontsize=9)
        
        # Généré
        gen_path = f'../results/generated_audio/hassaniya_gen_{i:03d}.mp3'
        if os.path.exists(gen_path):
            gen, _ = librosa.load(gen_path, sr=SAMPLE_RATE)
            mel_gen = librosa.feature.melspectrogram(y=gen, sr=SAMPLE_RATE, n_mels=80)
            mel_gen_db = librosa.power_to_db(mel_gen, ref=np.max)
            librosa.display.specshow(mel_gen_db, y_axis='mel', x_axis='time',
                                     sr=SAMPLE_RATE, hop_length=256, ax=axes[i, 1])
        axes[i, 1].set_title(f'Généré (gTTS)', fontsize=9)
    
    axes[0, 0].text(0.5, 1.15, 'Original Hassaniya', transform=axes[0, 0].transAxes,
                     ha='center', fontsize=12, fontweight='bold')
    axes[0, 1].text(0.5, 1.15, 'Généré (Référence)', transform=axes[0, 1].transAxes,
                     ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../results/spectrograms/comparison_grid.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Grille de comparaison sauvegardée dans results/spectrograms/comparison_grid.png')

## 8. Métriques d'Évaluation

### Comment la Qualité TTS est Mesurée

| Métrique | Type | Description |
|----------|------|-------------|
| **MOS** | Subjective | Score d'Opinion Moyen (note humaine de 1 à 5) |
| **MCD** | Objective | Distorsion Cepstrale Mel (plus bas = meilleur) |
| **PESQ** | Objective | Évaluation Perceptuelle de la Qualité Vocale |
| **STOI** | Objective | Intelligibilité Objective à Court Terme |

Pour notre MVP, nous calculons la **Distorsion Cepstrale Mel (MCD)** comme métrique préliminaire.

In [ ]:
def compute_mcd(ref_audio, syn_audio, sr=SAMPLE_RATE, n_mfcc=13):
    """Calculer la Distorsion Cepstrale Mel entre l'audio de référence et synthétisé."""
    # Calculer les MFCCs
    mfcc_ref = librosa.feature.mfcc(y=ref_audio, sr=sr, n_mfcc=n_mfcc)
    mfcc_syn = librosa.feature.mfcc(y=syn_audio, sr=sr, n_mfcc=n_mfcc)
    
    # Aligner les longueurs (utiliser la plus courte)
    min_len = min(mfcc_ref.shape[1], mfcc_syn.shape[1])
    mfcc_ref = mfcc_ref[:, :min_len]
    mfcc_syn = mfcc_syn[:, :min_len]
    
    # Calculer la MCD (exclure c0)
    diff = mfcc_ref[1:] - mfcc_syn[1:]
    mcd = np.mean(np.sqrt(2 * np.sum(diff ** 2, axis=0)))
    return mcd

# Calculer la MCD pour les paires disponibles
if GTTS_AVAILABLE:
    mcds = []
    print('=== Distorsion Cepstrale Mel (MCD) ===')
    print('(Plus bas = meilleur ; <8 dB est bon pour un TTS même locuteur)\n')
    
    for i in range(min(5, len(df))):
        orig = decode_original(df['audio'].iloc[i])
        gen_path = f'../results/generated_audio/hassaniya_gen_{i:03d}.mp3'
        
        if orig is not None and os.path.exists(gen_path):
            gen, _ = librosa.load(gen_path, sr=SAMPLE_RATE)
            mcd = compute_mcd(orig, gen)
            mcds.append(mcd)
            text = df['transcription'].iloc[i][:40]
            print(f'  Échantillon {i} : MCD = {mcd:.2f} dB — "{text}..."')
    
    if mcds:
        print(f'\n  MCD moyenne : {np.mean(mcds):.2f} dB')
        print(f'  Note : MCD élevée attendue — gTTS utilise l\'arabe standard, pas le hassaniya')
        print(f'  Un modèle affiné atteindrait une MCD significativement plus basse')

## 9. Résumé des Résultats

In [ ]:
print('=' * 65)
print('     TTS HASSANIYA — RÉSUMÉ DES RÉSULTATS DE LA DÉMO')
print('=' * 65)
print()
print('  État du Pipeline :')
print('  ├── Prétraitement du texte :       ✓ Terminé')
print('  ├── Tokenisation par caractères :  ✓ Terminé')
print('  ├── Prétraitement audio :          ✓ Terminé')
print('  ├── Extraction Mel Spectrogramme : ✓ Terminé')
print('  ├── TTS de référence (gTTS) :      ✓ Fonctionnel')
print('  ├── Architecture du modèle :       ✓ Définie')
print('  └── Apprentissage par transfert :  ◐ Stratégie définie, nécessite entraînement GPU')
print()
print('  Résultats Clés :')
print('  - gTTS produit de l\'arabe intelligible mais sans prosodie hassaniya')
print('  - Les différences spectrales confirment les caractéristiques dialectales')
print('  - 294 échantillons suffisants pour des expériences initiales d\'affinage')
print('  - L\'entraînement complet nécessite : ressources GPU + temps d\'entraînement étendu')
print()
print('  Prochaines Étapes Recommandées :')
print('  1. Collecter 2000+ enregistrements hassaniya supplémentaires')
print('  2. Affiner le modèle VITS sur un cluster GPU')
print('  3. Implémenter un dictionnaire de phonèmes hassaniya')
print('  4. Mener une évaluation MOS avec des locuteurs natifs')
print('=' * 65)

## Conclusion

Ce notebook a démontré un **pipeline TTS complet** pour l'arabe hassaniya :

1. **Synthèse de référence** : Génération de parole arabe avec gTTS
2. **Architecture** : Définition d'un modèle inspiré de Tacotron2 pour le hassaniya
3. **Apprentissage par transfert** : Démonstration de la stratégie gel/dégel pour l'affinage
4. **Comparaison** : Visualisation des différences entre parole originale et synthétisée
5. **Évaluation** : Calcul de la métrique MCD pour l'évaluation de la qualité

### Point Clé

Bien que le système de référence gTTS produise de la parole en arabe standard, un **modèle affiné**
apprendrait la **prosodie, l'intonation et la phonologie** spécifiques au hassaniya.
L'architecture et le pipeline sont prêts — ce qu'il faut :
- Plus de données (5000+ échantillons)
- Du temps d'entraînement GPU (plusieurs jours)
- Une évaluation par des locuteurs natifs

Ce MVP prouve la **faisabilité** de la construction d'un TTS hassaniya par apprentissage par transfert.